# Z-Score Normalization

[]

In [31]:
cd Downloads/movielens/preproc_eval/

[WinError 3] 지정된 경로를 찾을 수 없습니다: 'Downloads/movielens/preproc_eval/'
C:\Users\ASUS\Downloads\movielens\preproc_eval


C:\Users\ASUS\anaconda3\envs\tf-env\lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [32]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split

DATA_PATH = './data'
ratings = pd.read_csv(f'{DATA_PATH}/ratings.csv')
movies  = pd.read_csv(f'{DATA_PATH}/movies.csv')

SEED_LIST = [10, 21, 35, 42, 57, 60, 73, 88, 95, 101]
PERSONA_HEAVY_USER = 414
PERSONA_GENRE_SPECIALIST = 85

## 1. 전처리 함수 (Z-Score + 최소 평점 필터링)

In [33]:
def run_preprocessing(train_df, test_df):
    train = train_df.copy()
    test  = test_df.copy()

    # 1. 최소 5회 이상 평점 준 사용자만 사용 (페르소나는 강제 포함)
    user_counts = train['userId'].value_counts()
    valid_users = set(user_counts[user_counts >= 5].index)
    valid_users = valid_users.union({PERSONA_HEAVY_USER, PERSONA_GENRE_SPECIALIST})

    train = train[train['userId'].isin(valid_users)]
    test  = test[test['userId'].isin(valid_users)]

    # 2. Train에서만 사용자별 mean / std 계산
    user_stats = train.groupby('userId')['rating'].agg(['mean', 'std']).fillna(1.0)
    user_stats['std'] = user_stats['std'].replace(0, 1.0)   # std=0 방지

    # 3. Z-Score 변환 (Train)
    train = train.merge(user_stats, on='userId', suffixes=('', '_user'))
    train['z_rating'] = (train['rating'] - train['mean']) / train['std']

    # 4. Test도 동일한 통계량으로 변환 (Train에 없는 사용자는 제거 → cold-start 방지)
    test = test.merge(user_stats, on='userId', how='left')
    test = test.dropna(subset=['mean', 'std'])   # Train에 없는 사용자 제거
    test['z_rating'] = (test['rating'] - test['mean']) / test['std']

    return train, test, user_stats   # user_stats도 반환 (역변환에 필요)

## 2. 평가 함수

In [34]:
def rmse(y_true, y_pred): return np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2))
def mae(y_true, y_pred):  return np.mean(np.abs(np.array(y_true)-np.array(y_pred)))

def time_predictions(fn, pairs):
    s = time.perf_counter()
    preds = [fn(u,i) for u,i in pairs]
    e = time.perf_counter() - s
    return preds, e, (e/len(pairs))*1000

## 3. Multi-seed 실험 루프

In [35]:
results = []
best_reco = None

for idx, SEED in enumerate(SEED_LIST):
    print(f"\nRunning Seed {SEED} ({idx+1}/{len(SEED_LIST)})")

    train_raw, test_raw = train_test_split(ratings, test_size=0.2, random_state=SEED)
    train, test, user_stats = run_preprocessing(train_raw, test_raw)

    # Z-Score된 rating으로 행렬 구성
    ui = train.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)
    user_ids  = ui.index.tolist()
    movie_ids = ui.columns.tolist()
    ui_mat = ui.values

    user_to_idx = {u:i for i,u in enumerate(user_ids)}
    movie_to_idx = {m:i for i,m in enumerate(movie_ids)}
    
    # Item-Item 유사도 (dot product on Z-score centered matrix)
    def get_neighbors(midx, K=40):
        sims = np.dot(ui_mat.T, ui_mat[:, midx])
        sims[midx] = -np.inf
        topk = np.argsort(-sims)[:K]
        return [(int(j), float(sims[j])) for j in topk if sims[j] > 0]

    global_z_mean = train['z_rating'].mean()

    def predict_z(user_id, movie_id):
        if user_id not in user_to_idx or movie_id not in movie_to_idx:
            return global_z_mean
        uidx = user_to_idx[user_id]
        midx = movie_to_idx[movie_id]
        neighbors = get_neighbors(midx)
        num, den = 0.0, 0.0
        for j_idx, sim in neighbors:
            if ui.iat[uidx, j_idx] == 0: continue
            r_z = ui.iat[uidx, j_idx]
            num += sim * r_z
            den += abs(sim)
        return 0.0 if den == 0 else num / den

    # Test 샘플링 (최대 3000개)
    rng = np.random.default_rng(SEED)
    test_sample = test[['userId', 'movieId', 'rating', 'mean', 'std']].sample(n=min(3000, len(test)), random_state=SEED)

    pairs = list(zip(test_sample['userId'], test_sample['movieId']))
    z_preds, elapsed, avg_ms = time_predictions(predict_z, pairs)

    # 역변환해서 원래 스케일로 되돌리기
    original_preds = []
    for pred_z, row in zip(z_preds, test_sample.itertuples()):
        mean_u = row.mean
        std_u  = row.std
        original_preds.append(pred_z * std_u + mean_u)

    res = {
        'seed': SEED,
        'rmse': rmse(test_sample['rating'], original_preds),
        'mae' : mae(test_sample['rating'], original_preds),
        'avg_ms': avg_ms
    }
    results.append(res)

    # 첫 번째 시드에서만 추천 결과 저장
    if idx == 0:
        def recommend(user_id, topk=3):
            if user_id not in user_to_idx: return None
            watched = set(train[train['userId']==user_id]['movieId'])
            candidates = [m for m in movie_ids if m not in watched]
            scores = []
            for m in candidates:
                z = predict_z(user_id, m)
                mean_u = user_stats.loc[user_id, 'mean']
                std_u  = user_stats.loc[user_id, 'std']
                scores.append((m, z * std_u + mean_u))
            scores.sort(key=lambda x: -x[1])
            top = scores[:topk]
            return [{
                'movieId': m,
                'title': movies.loc[movies['movieId']==m, 'title'].iloc[0],
                'predicted_rating': round(score, 3)
            } for m, score in top]

        best_reco = {
            'heavy': recommend(PERSONA_HEAVY_USER),
            'specialist': recommend(PERSONA_GENRE_SPECIALIST)
        }

print("\n실험 완료!")


Running Seed 10 (1/10)

Running Seed 21 (2/10)

Running Seed 35 (3/10)

Running Seed 42 (4/10)

Running Seed 57 (5/10)

Running Seed 60 (6/10)

Running Seed 73 (7/10)

Running Seed 88 (8/10)

Running Seed 95 (9/10)

Running Seed 101 (10/10)

실험 완료!


## 4. 결과 요약

In [36]:
import pandas as pd
df = pd.DataFrame(results)
print(df[['seed','rmse','mae','avg_ms']].round(4))
print("\n=== 평균 ± 표준편차 ===")
print(f"RMSE : {df.rmse.mean():.4f} ± {df.rmse.std():.4f}")
print(f"MAE  : {df.mae.mean():.4f} ± {df.mae.std():.4f}")
print(f"예측 속도 : {df.avg_ms.mean():.2f} ms (± {df.avg_ms.std():.2f})")

print("\n=== 페르소나 추천 결과 (Seed 10) ===")
print(f"헤비 유저 (userId={PERSONA_HEAVY_USER})")
for x in best_reco['heavy']:
    print(f"  → {x['title']}  (예측 평점 {x['predicted_rating']})")

print(f"\n드라마 전문가 (userId={PERSONA_GENRE_SPECIALIST})")
for x in best_reco['specialist']:
    print(f"  → {x['title']}  (예측 평점 {x['predicted_rating']})")

   seed    rmse     mae  avg_ms
0    10  0.9312  0.6987  2.2204
1    21  0.9250  0.6981  1.6332
2    35  0.9383  0.7043  1.5169
3    42  0.9505  0.7039  1.6240
4    57  0.9041  0.6841  1.5342
5    60  0.9081  0.6863  1.5835
6    73  0.9206  0.6975  1.6644
7    88  0.9109  0.6897  1.5829
8    95  0.8877  0.6756  1.5974
9   101  0.9211  0.6916  1.5424

=== 평균 ± 표준편차 ===
RMSE : 0.9197 ± 0.0181
MAE  : 0.6930 ± 0.0092
예측 속도 : 1.65 ms (± 0.21)

=== 페르소나 추천 결과 (Seed 10) ===
헤비 유저 (userId=414)
  → Harmonists, The (1997)  (예측 평점 5.0)
  → Happy Go Lovely (1951)  (예측 평점 5.0)
  → Hawks and Sparrows (Uccellacci e Uccellini) (1966)  (예측 평점 5.0)

드라마 전문가 (userId=85)
  → Toy Story (1995)  (예측 평점 5.0)
  → Heat (1995)  (예측 평점 5.0)
  → American President, The (1995)  (예측 평점 5.0)
